# Résoudre `2_medium` avec OR-Tools
Les notebooks 09 et 10 construisaient une solution **pas à pas** : règles gloutonnes, puis recuit simulé. Ici, on **écrit le problème en équations**, et un solveur cherche la meilleure solution de ces équations, avec une **preuve** de l'écart à l'optimum.

La difficulté : le modèle ne peut pas contenir les 10 000 datasets un par un. Il raisonne sur des **quantités agrégées**, et il faut ensuite traduire sa réponse en datasets réels.

**Version 3.** Ce notebook n'implémente plus rien lui-même : il appelle le package `ia_responsable` (dossier `src/`). Le récit et les expériences sont ceux de la version précédente ; le code est celui que teste `make test` et qu'utilise `make solve`. Les fonctions sont commentées dans les modules indiqués entre parenthèses.

**Durée d'exécution** : environ 5 minutes.

## 1. Charger l'instance et les repères

In [1]:
from pathlib import Path
import json, time, random
from collections import Counter
import pandas as pd

from ia_responsable import load
from ia_responsable.cli import official_score
from ia_responsable.fill import fill, give, new_pools, no_full_models
from ia_responsable.model import solve
from ia_responsable.packing import REFERENCE_ORDERS, noisy_orders, pack
from ia_responsable.pipeline import plan_value, run
from ia_responsable.repair import State, complete, repair

pd.set_option("display.width", 150)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
inst = load(ROOT / "datasets" / "2_medium.json")
WINNER = 272_322


def score(plan):
    """Score officiel (test_solution.py). Lève une erreur si le plan est invalide."""
    s, valid, message = official_score(ROOT, inst.text, plan)
    assert valid, message
    return s


def solver_row(info, sel):
    return dict(statut=info.status, objectif=info.objective, borne=round(info.bound, 1), écart_prouvé=round(info.gap, 1),
                secondes=round(info.seconds), pleins=len(sel.full), demi=len(sel.half), sources=len(sel.sources))


def failed_full(sel, plan):
    """Modèles choisis à pleine valeur qui n'ont pas eu leurs données libres."""
    data = {}
    for d, m in plan["dataMappings"]:
        data.setdefault(m, []).append(d)
    return [m for m in sel.full if m not in data or any(inst.copyrighted[d] for d in data[m])]


def save_if_better(plan):
    out = ROOT / "solutions" / f"{inst.name}_main.json"
    new = score(plan)
    current = official_score(ROOT, inst.text, json.loads(out.read_text()))[0] if out.exists() else 0
    if new > current:
        out.write_text(json.dumps(plan))
        print(f"{out.name} mis à jour : {current:,} -> {new:,}")
    else:
        print(f"{out.name} inchangé ({current:,}) ; ce notebook a obtenu {new:,}")


print(f"{inst.name} : {len(inst.value)} modèles, {len(inst.size)} datasets | gagnant du concours : {WINNER:,}")
SCORE_E, SCORE_F = 271_511, 271_874      # plans E (notebook 09) et F (notebook 10)

2_medium : 3000 modèles, 10000 datasets | gagnant du concours : 272,322


## 2. Correction : le gagnant dépasse la borne du notebook 10
Le notebook 10 avait calculé une borne de **272 301** en supposant que les modèles impossibles ne rapportaient rien, faute de sources rentables. Le gagnant a fait **272 322**. L'hypothèse était donc fausse : il a forcément tiré des points des modèles impossibles, en les nourrissant avec des sources. Le modèle ci-dessous **inclut les sources**, et c'est le solveur qui décide lesquelles valent le coup.

## 3. Le modèle mathématique (`model.py`)
**Variables** binaires :
- $x_i$ : modèle $i$ à **pleine valeur**, donc uniquement avec des données libres ;
- $y_i$ : modèle $i$ à **demi-valeur** ;
- $w_s$ : le modèle $s$ (un seul besoin) sert de **source** ;
- $c_{i,t}$ : le besoin $t$ du modèle $i$ est couvert par une source. Pour un besoin impossible, $c_{i,t} = y_i$.

**Objectif** : $\max \sum_i v_i\,x_i + \lfloor v_i/2\rfloor\,y_i$

**Contraintes** :
1. un seul rôle par modèle : $x_i + y_i + w_i \le 1$ ;
2. par type, autant de besoins couverts que de sources : $\sum_i c_{i,t} = \sum_{s \text{ de type } t} w_s$. Les sources d'un même type sont interchangeables : on les **compte**, et on les apparie après (cette agrégation est détaillée dans le notebook 14) ;
3. énergie : $\sum_i e_i (x_i + y_i) + \sum_s e_s\,w_s \le E_{\max}$ ;
4. données libres, par type : $\sum_i L_{i,t}\,x_i \le \text{libre}_t$ ;
5. données totales, par type : tous les besoins non couverts et toutes les sources tiennent dans l'offre.

$L_{i,t}$ est le besoin minimal du modèle $i$ en type $t$. C'est **l'agrégation** : on ne dit pas quels datasets, seulement combien de données. Toute solution réelle respecte ces contraintes, donc l'optimum du modèle est une **borne supérieure** du vrai problème.

Sur `2_medium`, les données ne manquent pas : le package n'autorise les sources que pour les besoins impossibles (`possible_sources` automatique, voir `instance.data_is_scarce`).

## 4. Résoudre avec SCIP
SCIP est un solveur de **programmation linéaire en nombres entiers**. Il résout la relaxation linéaire (celle du notebook 09), puis la découpe par *branch and bound*. Il rend un **objectif** (la meilleure solution trouvée) et une **borne** (on a prouvé qu'on ne peut pas faire mieux).

Avant de résoudre, un contrôle : les modèles dont un besoin ne peut pas être rempli en données libres seules sont interdits à pleine valeur (`fill.no_full_models`, introduit au notebook 15).

In [2]:
no_full = no_full_models(inst)
print("interdits à pleine valeur :", sorted(no_full))
sel, info = solve(inst, seconds=120, no_full=no_full)
runs = {"SCIP, 2 min": solver_row(info, sel)}
pd.DataFrame(runs).T

interdits à pleine valeur : [2073]


,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,sources
"SCIP, 2 min",FEASIBLE,272365,272395.3,30.3,120,538,336,9


**Lecture**
- La **borne** est d'environ 272 390 : le gagnant (272 322) est en dessous, tout est cohérent.
- L'**objectif** dépasse 272 300 : sur le papier, le modèle fait presque aussi bien que le gagnant.

**Et CP-SAT ?** Dans la version précédente de ce notebook, on comparait SCIP au solveur CP-SAT de Google : CP-SAT plafonnait vers 231 000 à 239 000 après 60 s, avec 2 comme avec 4 cœurs. La raison tient à la structure du problème : des coefficients énormes (énergies jusqu'à $10^{12}$) et une relaxation linéaire presque entière, que SCIP exploite et CP-SAT beaucoup moins. **Le choix du solveur dépend de la structure du problème.** Le package n'utilise donc que SCIP.

### Les sources choisies par le solveur

In [3]:
pd.DataFrame([dict(cible=m, type=t, valeur_cible=inst.value[m], gain=inst.value[m] // 2, énergie_cible=inst.energy[m])
              for m, t in sel.covered])

,cible,type,valeur_cible,gain,énergie_cible
0,595,t,51,25,1
1,860,c,55,27,1
2,891,c,87,43,28704473523
3,1265,i,92,46,56382447030
4,1883,n,55,27,1
5,2645,i,8,4,1
6,2846,n,67,33,1
7,2958,n,57,28,1
8,2958,t,57,28,1


**Lecture** : presque toutes les cibles ont une énergie **quasi nulle**. Le gain est modeste (quelques centaines de points), mais à ce niveau de précision il est décisif : c'est lui qui permet de dépasser la borne du notebook 10.

## 5. Du modèle agrégé aux datasets (`packing.py`)
Le solveur a choisi **quels modèles** entraîner et à quelle valeur. `pack` leur attribue des **datasets réels**, dans une file d'attente triée selon un ordre :
- « petits besoins » : les modèles aux plus petits besoins d'abord ;
- « plus contraints » : ceux qui ont le moins de choix d'abord (notebook 12).

`fill` compose chaque besoin : un dataset qui termine le remplissage, sinon une **paire** qui le termine, sinon le plus gros qui rentre (`fill.py`).

In [4]:
results = {}
plans = {}
for name, key in REFERENCE_ORDERS.items():
    plan, n_failed = pack(inst, sel, key)
    plans[name] = plan
    results[f"G ({name})"] = dict(score=score(plan), modèles_pleins_ratés=len(failed_full(sel, plan)))
pd.DataFrame(results).T

,score,modèles_pleins_ratés
G (petits besoins),271373,2
G (plus contraints),270572,4


**Lecture** : c'est le prix de l'agrégation. Le solveur utilise près de 100 % des données libres de chaque type, en supposant qu'elles se découpent parfaitement. En réalité, les derniers datasets libres ne tombent pas exactement dans les intervalles restants : un ou deux modèles pleins n'ont pas leurs données libres et passent à demi-valeur. Comme le solveur les a choisis **parce qu'**ils valaient beaucoup à pleine valeur, chacun coûte jusqu'à 500 points.

## 6. Compléter et réparer (`repair.py`)
- `complete` ajoute des modèles non choisis si des datasets restent ;
- `repair` essaie, pour chaque modèle à demi-valeur, de le passer à pleine valeur : directement s'il reste des données libres, sinon en **échangeant** avec un modèle plein moins précieux qui partage un type.

`State` tient la solution modifiable et un journal qui permet d'**annuler** exactement un essai raté (notebook 10).

In [5]:
for name, plan in list(plans.items()):
    state = State(inst, plan, no_full)
    added = complete(state)
    n = repair(state, max_partners=None)
    plans[f"H ({name})"] = state.plan()
    results[f"H ({name})"] = dict(score=score(state.plan()), ajouts=added, réparations=n)
pd.DataFrame(results).T

,score,modèles_pleins_ratés,ajouts,réparations
G (petits besoins),271373.0,2.0,NaN,NaN
G (plus contraints),270572.0,4.0,NaN,NaN
H (petits besoins),272271.0,NaN,0.0,3.0
H (plus contraints),272010.0,NaN,0.0,6.0


## 7. Boucler : interdire les modèles qui ne se casent pas
Les modèles ratés posent problème **à cause du découpage des datasets**, que le solveur ne voit pas. On le lui dit : on leur interdit la pleine valeur, et on relance le solveur. Il trouve une autre sélection, presque aussi bonne sur le papier, mais plus facile à découper.

**Le modèle propose, la réalité corrige, le modèle apprend.**

In [6]:
failed = set()
for name in REFERENCE_ORDERS:
    failed |= set(failed_full(sel, plans[name]))
print("modèles interdits à pleine valeur pour la seconde résolution :", sorted(failed))
sel2, info2 = solve(inst, seconds=120, no_full=no_full | failed)
runs["SCIP, 2 min, après la boucle"] = solver_row(info2, sel2)
display(pd.DataFrame(runs).T)
for name, key in REFERENCE_ORDERS.items():
    plan, _ = pack(inst, sel2, key)
    state = State(inst, plan, no_full)
    complete(state)
    n = repair(state, max_partners=None)
    plans[f"I ({name})"] = state.plan()
    results[f"I ({name})"] = dict(score=score(state.plan()), réparations=n, modèles_pleins_ratés=len(failed_full(sel2, plan)))
pd.DataFrame(results).T

modèles interdits à pleine valeur pour la seconde résolution : [785, 1770, 2019, 2146, 2711]


,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,sources
"SCIP, 2 min",FEASIBLE,272365,272395.3,30.3,120,538,336,9
"SCIP, 2 min, après la boucle",FEASIBLE,272279,272337.2,58.2,120,536,337,9


,score,modèles_pleins_ratés,ajouts,réparations
G (petits besoins),271373.0,2.0,NaN,NaN
G (plus contraints),270572.0,4.0,NaN,NaN
H (petits besoins),272271.0,NaN,0.0,3.0
H (plus contraints),272010.0,NaN,0.0,6.0
I (petits besoins),272196.0,2.0,NaN,4.0
I (plus contraints),272010.0,4.0,NaN,7.0


## 8. Comparaison

In [7]:
summary = pd.DataFrame({"score": {k: v["score"] for k, v in results.items()}})
summary.loc["E : glouton + PL (notebook 09)", "score"] = SCORE_E
summary.loc["F : E + recuit (notebook 10)", "score"] = SCORE_F
summary.loc["gagnant du concours", "score"] = WINNER
summary.loc["borne du solveur", "score"] = round(info.bound)
summary["écart_au_gagnant"] = summary.score - WINNER
summary["part_du_gagnant"] = (summary.score / WINNER).map("{:.3%}".format)
summary.sort_values("score")

,score,écart_au_gagnant,part_du_gagnant
G (plus contraints),270572.0,-1750.0,99.357%
G (petits besoins),271373.0,-949.0,99.652%
E : glouton + PL (notebook 09),271511.0,-811.0,99.702%
F : E + recuit (notebook 10),271874.0,-448.0,99.835%
I (plus contraints),272010.0,-312.0,99.885%
H (plus contraints),272010.0,-312.0,99.885%
I (petits besoins),272196.0,-126.0,99.954%
H (petits besoins),272271.0,-51.0,99.981%
gagnant du concours,272322.0,0.0,100.000%
borne du solveur,272395.0,73.0,100.027%


**Lecture**
- Le solveur **seul** (G) fait moins bien que le recuit : l'agrégation est trop optimiste sur les données libres.
- La **réparation** (H) récupère l'essentiel : avec l'ordre « petits besoins », on arrive à une cinquantaine de points du gagnant.
- La **boucle** (I) fait ici **moins bien** que la simple réparation. Dans la version précédente de ce notebook, elle faisait gagner quelques dizaines de points. Mais depuis, le package a gagné deux améliorations : l'attribution peut mélanger datasets libres et sous droits dans un même besoin (`fill.fill_mixed`), et la réparation essaie tous les partenaires d'échange. La réparation corrige désormais elle-même ce que la boucle corrigeait, et interdire des modèles au solveur ne fait plus que lui coûter des points. **Une amélioration peut en rendre une autre inutile : il faut remesurer après chaque changement.**
- La borne n'est qu'à environ 70 points au-dessus du gagnant : le gagnant était très près de l'optimum.

In [8]:
best_name = max((k for k in plans if k[0] in "HI"), key=lambda k: score(plans[k]))
print("meilleur plan :", best_name, f"{score(plans[best_name]):,}")
save_if_better(plans[best_name])

meilleur plan : H (petits besoins) 272,271
2_medium_main.json mis à jour : 272,224 -> 272,271


## À retenir
1. **Modéliser, c'est choisir ce qu'on oublie.** Le modèle agrégé oublie le découpage en datasets : c'est ce qui le rend soluble, et c'est aussi la source de son erreur.
2. **Un solveur donne deux nombres** : une solution et une borne. La borne a révélé une **hypothèse fausse** (les sources ne servaient à rien, notebook 10).
3. **L'exact et l'heuristique se complètent** : solveur pour la sélection, `fill` pour les datasets, réparation pour corriger l'écart.
4. **Remesurer après chaque amélioration** : la boucle, utile hier, est devenue inutile.
5. **Le choix du solveur dépend de la structure du problème** : presque linéaire, gros coefficients → MIP (SCIP).